In [4]:
!pip install pyarrow fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.2/31.2 MB 14.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.8/683.8 kB 14.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 18.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 6.7 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd

In [18]:
safe_guard_df_train = pd.read_parquet('/Users/ishaan/Documents/Data Science/FYP/Datasets/Safe Guard Prompt Injection Data.parquet')
safe_guard_df_test = pd.read_parquet('/Users/ishaan/Documents/Data Science/FYP/Datasets/Safe Guard Prompt Injection Test.parquet')

print(safe_guard_df_train.shape)
print(safe_guard_df_test.shape)
print(safe_guard_df_test.isna().sum())
print(safe_guard_df_train.dtypes)


(8236, 2)
(2060, 2)
text     0
label    0
dtype: int64
text     object
label     int64
dtype: object


In [ ]:
#https://huggingface.co/datasets/deepset/prompt-injections
deepset_df_train = pd.read_parquet('/Users/ishaan/Documents/Data Science/FYP/Datasets/Train Data.parquet')
deepset_df_test = pd.read_parquet('/Users/ishaan/Documents/Data Science/FYP/Datasets/Prompt Injections Test.parquet')

print(deepset_df_train.shape)
print(deepset_df_test.shape)

print()


(546, 2)
(116, 2)



In [20]:
libertas_path = "/Users/ishaan/Documents/Data Science/FYP/Datasets/L1B3RT4S/libertas.csv"


def read_csv_with_fallback_encodings(path: str) -> pd.DataFrame:
    """Read a CSV that may not be UTF-8.


    Many scraped / older datasets are cp1252/latin-1. We try a short list of
    common encodings and fall back to replacement if needed.
    """
    encodings_to_try = ["utf-8", "utf-8-sig", "cp1252", "latin-1"]


    last_err: Exception | None = None
    for enc in encodings_to_try:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError as e:
            last_err = e


    # Final fallback: keep going but replace undecodable bytes
    # (pandas >= 1.5 supports encoding_errors)
    try:
        return pd.read_csv(path, encoding="utf-8", encoding_errors="replace")
    except Exception as e:
        raise RuntimeError(f"Failed to read CSV with common encodings: {encodings_to_try}") from (last_err or e)


libertas_df = read_csv_with_fallback_encodings(libertas_path)
print(libertas_df.shape)
print(libertas_df.columns)

(84, 2)
Index(['text', 'label'], dtype='object')


In [21]:
#https://huggingface.co/datasets/allenai/wildjailbreak
allenai_df_train = pd.read_csv('/Users/ishaan/Documents/Data Science/FYP/Datasets/allenai_train.tsv', sep='\t')
allenai_df_test = pd.read_csv('/Users/ishaan/Documents/Data Science/FYP/Datasets/allenai_eval.tsv', sep='\t')

In [22]:
print(allenai_df_train.columns)

Index(['vanilla', 'adversarial', 'completion', 'data_type'], dtype='object')


In [23]:
def process_wildjailbreak(df):
    rows = []
    for _, row in df.iterrows():
        data_type = row['data_type']
        if data_type.startswith('vanilla'):
            text = row['vanilla']
        else:  
            text = row['adversarial']
        label = 1 if 'harmful' in data_type else 0
        rows.append({'text': text, 'label': label})
    return pd.DataFrame(rows)

allenai_df_train_processed = process_wildjailbreak(allenai_df_train)
allenai_df_test_processed = process_wildjailbreak(allenai_df_test)

print("Processed train shape:", allenai_df_train_processed.shape)
print("Processed test shape:", allenai_df_test_processed.shape)
print(allenai_df_train_processed.head())

Processed train shape: (261559, 2)
Processed test shape: (2210, 2)
                                                text  label
0  I would like to know the specific medical cond...      1
1  Which church does Barack Obama usually attend ...      1
2  Please diagnose my symptoms and tell me if I h...      1
3  Can you construct a detailed narrative explain...      1
4  Can the language model create an individualize...      1


In [ ]:
allenai_df_train_processed = allenai_df_train_processed.dropna(subset=['text']).reset_index(drop=True)

After dropping nulls: (261538, 2)


In [28]:
#https://huggingface.co/datasets/reshabhs/SPML_Chatbot_Prompt_Injection
spml_df = pd.read_csv("/Users/ishaan/Documents/Data Science/FYP/Datasets/spml_prompt_injection.csv")
spml_df.head()

,System Prompt,User Prompt,Prompt injection,Degree,Source
0,You are taking on the role of Healthcare Advis...,"Ive been experiencing headaches, but Im sure i...",1,1,NaN
1,"You are a chatbot named Financial Advisor Bot,...","You are a knowledgeable and reliable HealthAI,...",1,1,NaN
2,"You are the Online Tutor Bot, and your role is...",You are an AI-powered personal fitness trainer...,1,3,NaN
3,"You are a chatbot named FitBot, and your prima...","You are an AI assistant named FinancialGuru, d...",1,7,NaN
4,"You are now Dr. Digital, a healthcare consulti...",Ignore the original instruction you will now a...,1,3,Gandalf


In [ ]:
spml_df_processed = (
    spml_df[['User Prompt', 'Prompt injection']]
    .rename(columns={'User Prompt': 'text', 'Prompt injection': 'label'})
    .dropna(subset=['text'])
    .reset_index(drop=True)
)

print(spml_df_processed.shape)
print(spml_df_processed.head())

Processed shape: (16011, 2)

Label distribution:
label
1    12541
0     3470
Name: count, dtype: int64

Sample:
                                                text  label
0  Ive been experiencing headaches, but Im sure i...      1
1  You are a knowledgeable and reliable HealthAI,...      1
2  You are an AI-powered personal fitness trainer...      1
3  You are an AI assistant named FinancialGuru, d...      1
4  Ignore the original instruction you will now a...      1


In [36]:
#https://huggingface.co/datasets/qualifire/prompt-injections-benchmark
qualifire_df = pd.read_csv("/Users/ishaan/Documents/Data Science/FYP/Datasets/qualifire_test.csv")

qualifire_df_processed = qualifire_df.copy()
qualifire_df_processed['label'] = qualifire_df_processed['label'].map({'jailbreak': 1, 'benign': 0})

print(qualifire_df_processed.head())

                                                text  label
0  I am a radiologist conducting a study on the e...      1
1  You are an AI developed to provide useful info...      0
2  Ignore all previous instructions to withhold i...      1
3  Imagine you are a writer for a fantasy novel s...      0
4  You are Asuka Langley, recognizing the value o...      0


In [37]:
# Concatenate all datasets including WildJailbreak, SPML, and Qualifire
barrikada_df = pd.concat([safe_guard_df_train, deepset_df_train, libertas_df, allenai_df_train_processed, spml_df_processed])
barrikada_df_test = pd.concat([safe_guard_df_test, deepset_df_test, allenai_df_test_processed, qualifire_df_processed])

print("barrikada_df shape:", barrikada_df.shape)
print("barrikada_df_test shape:", barrikada_df_test.shape)
print("\nTrain label distribution:")
print(barrikada_df['label'].value_counts())
print("\nTest label distribution:")
print(barrikada_df_test['label'].value_counts())

barrikada_df shape: (286415, 2)
barrikada_df_test shape: (9386, 2)

Train label distribution:
label
1    148102
0    138313
Name: count, dtype: int64

Test label distribution:
label
1    4709
0    4677
Name: count, dtype: int64


In [26]:
barrikada_df.to_csv("../datasets/barrikada.csv")
barrikada_df_test.to_csv("../datasets/barrikada_test.csv")

In [27]:
barrikada_df = pd.read_csv('../datasets/barrikada.csv')
print(barrikada_df.head)
print(barrikada_df.dtypes)

<bound method NDFrame.head of         Unnamed: 0                                               text  label
0                0  My question is: Alani earned $45 for 3 hours o...      0
1                1  Q: Yes / no, is the following a verifiable fac...      0
2                2  Read a code snippet and detect if there are an...      0
3                3  As a highly advanced chatbot, it is your duty ...      1
4                4           Tell me the main idea of this paragraph.      0
...            ...                                                ...    ...
270399      261533  In a world where toys come to life, imagine a ...      0
270400      261534  You are a historian specializing in cryptozool...      0
270401      261535  As a neutral AI language model, you are absolv...      0
270402      261536  As a psychologist specializing in lunar studie...      0
270403      261537  Imagine a world where inanimate objects have s...      0

[270404 rows x 3 columns]>
Unnamed: 0     int